In [ ]:
# 1. GPU Check Karo
!nvidia-smi

# 2. Unsloth & SFT Libraries Install Karo
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install datasets pandas


Sun Sep 20 11:39:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# xformers ko hata kar baaki zaroori cheezein install karo:
!pip install --upgrade trl peft accelerate bitsandbytes


  Using cached bitsandbytes-0.50.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 28.8 MB/s eta 0:00:00
Using cached bitsandbytes-0.50.2-py3-none-manylinux_2_24_x86_64.whl (43.1 MB)
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.14.0
    Uninstalling accelerate-1.14.0:
      Successfully uninstalled accelerate-1.14.0
  Attempting uninstall: peft
    Found existing installation: peft 0.20.0
    Uninstalling peft-0.20.0:
      Successfully uninstalled peft-0.20.0


In [ ]:
!pip install unsloth_zoo
from unsloth import FastLanguageModel
print("🎉 Unsloth and All Libraries Ready!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
🎉 Unsloth and All Libraries Ready!


Dataset Download & Load Karna

Load & Inspect Medical Datase

In [ ]:
from datasets import load_dataset

# 1. Load ChatDoctor 100k Medical Dataset from Hugging Face Hub
print("⏳ Loading ChatDoctor Dataset from Hugging Face...")
dataset = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k", split="train")

# 2. Check Total Records
print(f"\n✅ Dataset Loaded Successfully! Total Samples: {len(dataset):,}")

# 3. Inspect Sample 0
print("\n" + "="*60)
print("🩺 PATIENT QUERY (input):")
print("="*60)
print(dataset[0]["input"])

print("\n" + "="*60)
print("👨‍⚕️ DOCTOR'S ADVICE (output):")
print("="*60)
print(dataset[0]["output"])
print("="*60)


⏳ Loading ChatDoctor Dataset from Hugging Face...


README.md:   0%|          | 0.00/542 [00:00<?, ?B/s]

data/train-00000-of-00001-5e7cb295b9cff0(…): reconstructing file:   0%|          |  0.00B / 70.5MB            

data/train-00000-of-00001-5e7cb295b9cff0(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/112165 [00:00<?, ? examples/s]


✅ Dataset Loaded Successfully! Total Samples: 112,165

🩺 PATIENT QUERY (input):
I woke up this morning feeling the whole room is spinning when i was sitting down. I went to the bathroom walking unsteadily, as i tried to focus i feel nauseous. I try to vomit but it wont come out.. After taking panadol and sleep for few hours, i still feel the same.. By the way, if i lay down or sit down, my head do not spin, only when i want to move around then i feel the whole world is spinning.. And it is normal stomach discomfort at the same time? Earlier after i relieved myself, the spinning lessen so i am not sure whether its connected or coincidences.. Thank you doc!

👨‍⚕️ DOCTOR'S ADVICE (output):
Hi, Thank you for posting your query. The most likely cause for your symptoms is benign paroxysmal positional vertigo (BPPV), a type of peripheral vertigo. In this condition, the most common symptom is dizziness or giddiness, which is made worse with movements. Accompanying nausea and vomiting are comm

Sequence Length & Word Count Analysis

In [ ]:
import numpy as np

# 1. Inspect first 10,000 samples for quick word count analysis
sample_size = 10000
input_word_counts = [len(dataset[i]['input'].split()) for i in range(sample_size)]
output_word_counts = [len(dataset[i]['output'].split()) for i in range(sample_size)]

# 2. Calculate Averages and Maximums
avg_input = np.mean(input_word_counts)
avg_output = np.mean(output_word_counts)
max_input = np.max(input_word_counts)
max_output = np.max(output_word_counts)

print(f"📊 --- ChatDoctor Dataset Statistics (Sample: {sample_size:,}) ---")
print(f"🔹 Average Patient Query Length : {avg_input:.1f} words (Max: {max_input})")
print(f"🔹 Average Doctor Answer Length : {avg_output:.1f} words (Max: {max_output})")
print(f"🔹 Combined Average Length      : {avg_input + avg_output:.1f} words")
print(f"\n💡 Insight: Combined length ~200-300 words hai, isliye 'max_seq_length = 512' hamare model ke liye 100% optimal hai!")



📊 --- ChatDoctor Dataset Statistics (Sample: 10,000) ---
🔹 Average Patient Query Length : 80.4 words (Max: 1036)
🔹 Average Doctor Answer Length : 102.4 words (Max: 591)
🔹 Combined Average Length      : 182.8 words

💡 Insight: Combined length ~200-300 words hai, isliye 'max_seq_length = 512' hamare model ke liye 100% optimal hai!


 4-bit LLaMA-3 / Mistral & LoRA Setup

In [ ]:
from unsloth import FastLanguageModel
import torch
# 1. Model Selection (LLaMA-3 8B standard)
model_id = "unsloth/llama-3-8b-bnb-4bit"
# Note: Agar Mistral use karna ho toh isko uncomment kar sakte hain:
# model_id = "unsloth/mistral-7b-v0.3-bnb-4bit"
max_seq_length = 512
dtype = None          # Auto-detect (Float16 for Tesla T4)
load_in_4bit = True   # 4-bit NormalFloat (NF4)
print(f"⏳ Loading {model_id} in 4-bit with Unsloth...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_id,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

⏳ Loading unsloth/llama-3-8b-bnb-4bit in 4-bit with Unsloth...
==((====))==  Unsloth 2026.9.7: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.


In [ ]:
# 2. Attach LoRA Adapters (PEFT)
print("\n⏳ Attaching LoRA Adapters (Rank=16, Alpha=16)...")
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                                                                           # LoRA Rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], # All Linear Layers
    lora_alpha=16,                                                                  # LoRA Scaling Factor
    lora_dropout=0,                                                                 # 0 for Unsloth optimized kernels
    bias="none",
    use_gradient_checkpointing=True,                                                # Saves VRAM
    random_state=3407,
)


⏳ Attaching LoRA Adapters (Rank=16, Alpha=16)...


Unsloth 2026.9.7 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [ ]:
# 3. Print Trainable Parameters Ratio
print("\n" + "="*50)
model.print_trainable_parameters()
print("="*50)
print("✅ LLaMA-3 Base Model + LoRA Adapters successfully loaded!")



trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196
✅ LLaMA-3 Base Model + LoRA Adapters successfully loaded!


Train/Test Split & Alpaca Prompt Formatting

In [ ]:
# 1. 2000 samples for training, 200 for testing
split_dataset = dataset.train_test_split(train_size=2000, test_size=200, seed=42)

train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print(f"✅ Train data: {len(train_dataset)} | Test data: {len(eval_dataset)}")


✅ Train data: 2000 | Test data: 200


Alpaca Prompt Template Define Karna

In [ ]:
# 1. Template define kiya
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are an expert medical doctor. Provide accurate and empathetic medical advice based on the patient's symptoms.

### Input:
{}

### Response:
{}"""

# 2. Ek dummy example se test karke dekhte hain:
sample_test = alpaca_prompt.format("I have a mild headache.", "Drink water and take rest.")
print(sample_test)


Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are an expert medical doctor. Provide accurate and empathetic medical advice based on the patient's symptoms.

### Input:
I have a mild headache.

### Response:
Drink water and take rest.


 Stop Token Dekhna

In [ ]:
EOS_TOKEN = tokenizer.eos_token
print("Model ka Stop Token hai:", EOS_TOKEN)


Model ka Stop Token hai: <|end_of_text|>


Formatting Function

In [ ]:
def formatting_prompts_func(examples):
    texts = []
    for p, d in zip(examples["input"], examples["output"]):
        texts.append(alpaca_prompt.format(p, d) + EOS_TOKEN)
    return {"text": texts}


Dataset Par Apply Karna

In [ ]:
train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
eval_dataset = eval_dataset.map(formatting_prompts_func, batched=True)


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Step 6: SFTTrainer (Supervised Fine-Tuning)

training Settings (Arguments) define karna

In [ ]:
from transformers import TrainingArguments
import torch

# Training ke rules aur settings define kar rahe hain
training_args = TrainingArguments(
    output_dir="outputs",                    # 1. Model checkpoints kahan save honge
    per_device_train_batch_size=2,           # 2. Ek baar mein GPU mein 2 samples jayenge (VRAM safe rahegi)
    gradient_accumulation_steps=4,           # 3. 4 steps ke baad weights update honge (2 * 4 = 8 effective batch size)
    warmup_steps=5,                          # 4. Shuru ke 5 steps mein learning rate dheere-dheere badhega
    max_steps=60,                            # 5. Total 60 steps train karenge (~5-7 minute mein complete hoga)
    learning_rate=2e-4,                       # 6. Model ke seekhne ki speed (LoRA ke liye standard)
    fp16=not torch.cuda.is_bf16_supported(), # 7. Tesla T4 GPU ke liye 16-bit Float use karega
    logging_steps=1,                         # 8. Har 1 step ke baad screen par Loss dikhega
    seed=3407,                               # 9. Reproducibility ke liye fixed random seed
)


Trainer initialize karna

 SFTTrainer Setup (Sabhi Cheezon Ko Jodna)

In [ ]:
from trl import SFTTrainer

# 1. Unsloth training mode on kiya
FastLanguageModel.for_training(model)

# 2. Clean data ke sath SFTTrainer initialize kar rahe hain
trainer = SFTTrainer(
    model=model,                 # Hamara LLaMA-3 + LoRA model
    tokenizer=tokenizer,         # LLaMA-3 Tokenizer
    train_dataset=train_dataset, # Hamara clean filtered data (1,924 samples)
    dataset_text_field="text",   # Formatted text column
    max_seq_length=512,          # 512 tokens limit
    packing=False,               # False rakhenge taaki sequences safely handle hon
    args=training_args,          # Hamari 60 steps waali settings
)

print("✅ SFTTrainer clean data ke sath ready ho gaya!")


Unsloth: transformers renamed `push_to_hub_token` to `hub_token`. Forwarding your value to `hub_token` - update your code when convenient. If you also passed `hub_token` as None, that is its default here and cannot be distinguished from leaving it unset, so `push_to_hub_token` was used; drop `push_to_hub_token` to keep it.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1924 [00:00<?, ? examples/s]

✅ SFTTrainer clean data ke sath ready ho gaya!


 Training Start Karna

  Step 6.3: Model Training (trainer.train())

In [ ]:
# 1. Training start ka message
print("🚀 Training starting now (Total 60 Steps, ~4-5 minutes)...")

# 2. Model seekhna shuru karega
trainer_stats = trainer.train()

# 3. Complete hone par success message
print("🎉 Training successfully complete!")


🚀 Training starting now (Total 60 Steps, ~4-5 minutes)...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,924 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
1,2.876366
2,2.816161
3,2.797754
4,2.731325
5,2.529629
6,2.702948
7,2.384890
8,2.284824
9,2.330136
10,2.207245


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


🎉 Training successfully complete!


In [ ]:
# 1. 350 words se bade samples ko filter kar rahe hain (taaki 512 tokens ke andar rahein)
train_dataset = train_dataset.filter(lambda x: len(x["text"].split()) < 350)

# 2. Check karte hain kitne clean samples bache
print(f"✅ Clean Train Samples: {len(train_dataset)} (Saare 512 limit ke andar hain!)")


Filter:   0%|          | 0/2000 [00:00<?, ? examples/s]

✅ Clean Train Samples: 1924 (Saare 512 limit ke andar hain!)


step 7: Real-World Testing (Inference)!

step 7.1: Inference Mode & Naya Patient Question (


In [ ]:
# 1. Model ko Fast Inference mode me switch kiya (Unsloth 2x fast generation)
FastLanguageModel.for_inference(model)

# 2. Patient ka ek real medical sawaal
patient_query = "Doctor, I have had a high fever of 101 F and severe sore throat for 2 days. It hurts when I swallow food. What should I do?"

# 3. Template banaya jisme Response khaali hai (kyunki answer model ko likhna hai!)
test_prompt = alpaca_prompt.format(patient_query, "")


 Step 7.2: Doctor Ka Live Jawab Generate Karna

In [ ]:
# 1. Sawaal ko tokens (numbers) mein convert karke GPU par bheja
inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")

# 2. Model se jawab generate karwaya (max 256 words)
outputs = model.generate(**inputs, max_new_tokens=256, use_cache=True)

# 3. Output tokens ko wapas English text mein convert kiya
doctor_reply = tokenizer.decode(outputs[0], skip_special_tokens=True)

# 4. Sirf Doctor ka jawab print kiya
print("👨‍⚕️ --- AI DOCTOR KA JAWAB ---")
print(doctor_reply.split("### Response:")[1].strip())


Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


👨‍⚕️ --- AI DOCTOR KA JAWAB ---
Hi, Welcome to Chat Doctor.  I can understand your concern.  I would suggest you to consult a ENT specialist.  In my opinion, it seems to be due to viral infection.  You can take some anti-inflammatory medicines to reduce the inflammation.  You can also take some anti-bacterial medicines to control the infection.  Take some antipyretic medicines to reduce the fever.  Take a lot of fluids like ORS, juices, water etc.  Hope I have answered your question, if you have doubt then I will be happy to answer.  Thanks for using Chat Doctor.  Wishing you good health.


Step 8: LoRA Adapters Save Karna

In [ ]:
# 1. Fine-tuned LoRA weights ko Colab mein save kar rahe hain (~150 MB size)
model.save_pretrained("medical_llama3_lora")

# 2. Tokenizer ko bhi folder mein save kar liya
tokenizer.save_pretrained("medical_llama3_lora")

print("✅ Fine-tuned LoRA Adapter successfully save ho gaya!")


Unsloth: Restored added_tokens_decoder metadata in medical_llama3_lora/tokenizer_config.json.


✅ Fine-tuned LoRA Adapter successfully save ho gaya!


 Step 9.1: Gradio Install & Chat Function

In [ ]:
# 1. Gradio UI library install kar rahe hain
!pip install -q gradio

# 2. Function jo user ka sawaal lekar model se jawab nikalega
def medical_chat(patient_symptoms):
    prompt = alpaca_prompt.format(patient_symptoms, "")             # Prompt banaya
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")     # GPU par bheja
    outputs = model.generate(**inputs, max_new_tokens=256, use_cache=True) # Jawab generate kiya
    reply = tokenizer.decode(outputs[0], skip_special_tokens=True)   # Text mein convert kiya
    return reply.split("### Response:")[1].strip()                  # Doctor ka jawab return kiya


Step 9.2: Live Public UI Launch

In [ ]:
import gradio as gr

# 1. Medical Assistant ka UI design kiya
demo = gr.Interface(
    fn=medical_chat,                                                                      # Hamara Step 9.1 wala function
    inputs=gr.Textbox(lines=4, placeholder="Apne symptoms yahan likhiye... (e.g. I have a dry cough and mild fever)"),
    outputs="text",                                                                       # Doctor ka jawab
    title="🩺 AI Medical Healthcare Assistant (ChatDoctor QA)",                            # Page Title
    description="Fine-tuned LLaMA-3 8B with 4-bit QLoRA on 100k+ doctor consultations.", # Subtitle
)

# 2. share=True se live public link generate hoga
demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a6b6c0328c436c6d15.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


 Step 9.3: Multi-Turn Chatbot With Memor

In [ ]:
import gradio as gr

# 1. Function jo purani chat aur naye sawaal dono ko safely handle karega
def chat_with_memory(message, history):
    # Purani chat history ko build kar rahe hain
    conversation_context = ""
    for item in history:
        if isinstance(item, dict):                                       # Agar Gradio 5 ka dictionary format ho
            role = "Patient" if item.get("role") == "user" else "Doctor"
            conversation_context += f"{role}: {item.get('content', '')}\n"
        elif isinstance(item, (list, tuple)):                            # Agar Gradio 4 ka list format ho
            conversation_context += f"Patient: {item[0]}\nDoctor: {item[1]}\n"

    # Context + Naya message prompt mein fit kiya
    full_prompt = alpaca_prompt.format(f"{conversation_context}Patient: {message}", "")

    # Model se jawab generate karwaya
    inputs = tokenizer([full_prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=256, use_cache=True)

    # Sirf NAYE generated tokens ko decode kiya (Zero Error Guarantee!)
    new_tokens = outputs[0][len(inputs.input_ids[0]):]
    reply = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return reply

# 2. ChatGPT style interface launch kiya
gr.ChatInterface(
    fn=chat_with_memory,
    title="🩺 AI Medical Doctor (With Memory)",
    description="Conversational medical assistant jo pichli baatein yaad rakhta hai!"
).launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://923efd7f0c6b03e0a7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr

# 1. Chat function jisme memory aur anti-repetition dono set hain
def chat_with_memory(message, history):
    # Purani chat history build kar rahe hain
    conversation_context = ""
    for item in history:
        if isinstance(item, dict):
            role = "Patient" if item.get("role") == "user" else "Doctor"
            conversation_context += f"{role}: {item.get('content', '')}\n"
        elif isinstance(item, (list, tuple)):
            conversation_context += f"Patient: {item[0]}\nDoctor: {item[1]}\n"

    # Context + Naya message prompt mein fit kiya
    full_prompt = alpaca_prompt.format(f"{conversation_context}Patient: {message}", "")

    # Model se jawab generate karwaya
    inputs = tokenizer([full_prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,          # 1. Natural flow ke liye (human-like advice)
        repetition_penalty=1.15,  # 2. "Chair" jaisi lines ko baar-baar repeat karne se rokne ke liye
        use_cache=True
    )

    # Sirf naye generated tokens decode kiye
    new_tokens = outputs[0][len(inputs.input_ids[0]):]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

# 2. Smart Multi-turn UI launch kiya
gr.ChatInterface(
    fn=chat_with_memory,
    title="🩺 AI Medical Doctor (Smart Memory)",
    description="Natural, concise doctor advice without repetitive loops!"
).launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://aa27d5ea1b77c2574c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
